# 02 · Signal exploration

## Purpose
Look at the signals before any feature exists, and set the few physics parameters Stage 2 needs:
the microphone burst threshold (and prove bursts are an artefact worth removing), the envelope demodulation band per
radial accelerometer, and the bearing comb definitions. It also checks the one recording whose speed estimates
disagree, and audits whether `cage_fault` really looks like a cage fault.

Rule 6 applies throughout: only **train** recordings (`split_closed == "train"`) set a parameter. Train + val are used
for descriptive checks. Test recordings are never loaded.

## Inputs
`metadata.csv`, `data/raw_npy/`, `configs/base.yaml`, and the caches written by `scripts/03_signal_scan.py`
(named by the sha256 of the config sections that produced them).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tools this notebook needs, reads <code>metadata.csv</code> and the settings, and finds the cached results that <code>scripts/03_signal_scan.py</code> computed earlier. If a cache is missing, it stops and says which scan to run. It also defines two small helpers: one loads a recording, one measures how far each sample is from normal (the robust z-score).</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, welch
from scipy.stats import kruskal, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score

from ml.config import config_sha256, load_config
from ml.features import amplitude_spectrum, comb_lines, line_prominence_db, order_spectrum, select_demod_band
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, cache_file, data_dir
from ml.preprocess import remove_mic_bursts, speed_candidates, tacho_intervals

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
FS = pre.fs_hz
CH = list(pre.channels)
MIC = CH.index("mic")
ACCELS = [CH.index(c) for c in CH if c not in ("tacho", "mic")]
FIG = SYSTEM_CODE_DIR / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
train = meta[meta.split_closed == "train"]
trval = meta[meta.split_closed != "test"]

caches = {
    "mic": cache_file("02_mic", pre.mic, ext="parquet"),
    "meanspec": cache_file("02_meanspec", pre.mic, ext="npz"),
    "sk": cache_file("02_sk", pre.envelope.sk, ext="parquet"),
    "envelope": cache_file("02_envelope", pre.envelope, ext="npz"),
    "sensor": cache_file("02_sensor", ext="parquet"),
    "bandcompare": cache_file("02_bandcompare", pre.envelope, ext="npz"),
    "micalign": cache_file("02_micalign", pre.mic, ext="npz"),
    "tone": cache_file("02_tone", pre.mic, ext="parquet"),
    "shaft": cache_file("02_shaft", ext="parquet"),
}
missing = [k for k, p in caches.items() if not p.exists()]
assert not missing, f"run: python scripts/03_signal_scan.py {' '.join(missing)}"


def load(rec_id: str) -> np.ndarray:
    return np.load(data_dir() / "raw_npy" / f"{rec_id}.npy")


def robust_z(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    return np.abs(x - med) / (1.4826 * np.median(np.abs(x - med)))


print(f"preprocess config sha256 {config_sha256(pre)[:16]}…")
for k, p in caches.items():
    print(f"cache {k:9s} {p.name}")

## (a) Microphone bursts

### Where the bursts are
`peakz_mic` is the largest robust z-score |x − median| / (1.4826 · MAD) of the whole microphone recording, computed at
indexing. A burst-free microphone stays in single digits. Log scale, train + val, one panel per class, so sub-threshold
bursts in other classes would show as a tail.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws one histogram per class of each recording's largest microphone spike score, on a log scale. Recordings with bursts sit far to the right; the red dashed line is the burst threshold.</div>

In [ ]:
bins = np.logspace(np.log10(2), np.log10(3000), 60)
fig, axes = plt.subplots(2, 5, figsize=(15, 5.5), sharex=True)
for ax, cls in zip(axes.flat, CLASS10):
    v = trval.loc[trval.class10 == cls, "peakz_mic"]
    ax.hist(v, bins=bins, color="#4c72b0")
    ax.axvline(pre.mic.burst_z, color="red", lw=1, ls="--")
    ax.set(xscale="log", title=f"{cls} (n={len(v)})")
for ax in axes[-1]:
    ax.set_xlabel("peak robust z of microphone")
for ax in axes[:, 0]:
    ax.set_ylabel("recordings")
fig.suptitle(f"Microphone peak robust z, train + val (red: burst threshold {pre.mic.burst_z})")
fig.tight_layout()
fig.savefig(FIG / "02_mic_peakz_hist.png")

### The threshold
Rule: **z_thresh = 1.25 × the largest `peakz_mic` among `split_closed` TRAIN normal recordings**, rounded to 0.1.
Train normal recordings define "no bursts"; the 25 % margin keeps ordinary loud moments out. The check below confirms the
value falls in the gap between burst-free and bursty files, and lists exactly which files it catches (all splits,
from the indexed statistic; no test signal is loaded).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Works out the burst threshold from the rule (1.25 × the largest spike score among training normal recordings) and checks it equals the value stored in the settings. It then counts which files the threshold catches, per class and severity, and saves the full list to <code>results/mic_burst_files.csv</code>.</div>

In [ ]:
tn = train.loc[train.class10 == "normal", "peakz_mic"]
rule_z = round(1.25 * tn.max(), 1)
print(f"train normal: n={len(tn)}, max {tn.max():.2f}, p95 {tn.quantile(.95):.2f} → rule z_thresh {rule_z}")
assert rule_z == pre.mic.burst_z, "configs/base.yaml mic.burst_z is out of date"

below = meta.loc[meta.peakz_mic <= pre.mic.burst_z, "peakz_mic"]
caught = meta[meta.peakz_mic > pre.mic.burst_z].sort_values(["class10", "severity", "fr_file_hz"])
print(f"gap: largest peakz below threshold {below.max():.2f}, smallest above {caught.peakz_mic.min():.2f}")
print(f"caught: {len(caught)} files ({(caught.split_closed != 'test').sum()} train+val)")
caught[["rec_id", "class10", "severity", "split_closed", "peakz_mic"]].to_csv(
    SYSTEM_CODE_DIR / "results" / "mic_burst_files.csv", index=False)
pd.crosstab([caught.class10, caught.severity], caught.split_closed, margins=True)

The weakest caught files, and the loudest burst-free ones on either side of the gap:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the three loudest recordings that stay under the threshold next to the three quietest ones above it, so you can see the gap the threshold sits in.</div>

In [ ]:
pd.concat([meta.loc[below.sort_values().index[-3:], ["rec_id", "class10", "split_closed", "peakz_mic"]],
           caught.nsmallest(3, "peakz_mic")[["rec_id", "class10", "split_closed", "peakz_mic"]]])

### What a burst looks like
Three train bursts at sample level: the largest burst in one imbalance file, one underhang cage file and the lone overhang
cage file, ±2 ms around it. An electrical glitch lasts a few samples, has no ring-down, and has no matching accelerometer
response; an acoustic event would ring and would usually shake the frame too.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Zooms in on one burst in each of three recordings, 2 ms either side. The top row is the microphone; the bottom row is the six accelerometers. If the bursts were real sounds from the machine, the accelerometers would react too.</div>

In [ ]:
burst_train = trval[(trval.peakz_mic > pre.mic.burst_z)]
examples = [burst_train[burst_train.class10 == "imbalance"].sort_values("peakz_mic").iloc[len(burst_train) // 4].rec_id,
            burst_train[burst_train.class10 == "underhang_cage"].sort_values("peakz_mic").iloc[20].rec_id]
oc = burst_train[burst_train.class10 == "overhang_cage"]
examples.append(oc.rec_id.iloc[0] if len(oc) else burst_train.sort_values("peakz_mic").rec_id.iloc[-1])

half = int(0.002 * FS)
fig, axes = plt.subplots(2, 3, figsize=(15, 5.5), sharex=True)
for j, rid in enumerate(examples):
    x = load(rid)
    k = int(np.argmax(np.abs(x[:, MIC] - np.median(x[:, MIC]))))
    sl = slice(max(k - half, 0), k + half)
    t = (np.arange(x.shape[0])[sl] - k) / FS * 1e3
    axes[0, j].plot(t, x[sl, MIC], lw=0.8, color="#8172b2")
    axes[0, j].set(title=rid, ylabel="microphone (V)")
    for c in ACCELS:
        axes[1, j].plot(t, robust_z(x[:, c])[sl], lw=0.6, label=CH[c])
    axes[1, j].axhline(pre.mic.burst_z, color="red", ls="--", lw=0.8)
    axes[1, j].set(xlabel="time from burst peak (ms)", ylabel="accelerometer robust z")
axes[1, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG / "02_mic_burst_zoom.png")

The same questions over every train + val burst file (from the scan cache): run length in samples, the largest
accelerometer robust z within ±2 ms of each burst against the same statistic at 50 random times, and the microphone RMS
just after the burst relative to the whole recording (ring-down ≫ 1 would mean an acoustic event).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Summarises every burst recording from the cache: how many bursts, how long they last (in samples), how much the accelerometers react at a burst compared with random moments, and whether the microphone keeps ringing afterwards.</div>

In [ ]:
mic = pd.read_parquet(caches["mic"])
b = mic[mic.n_bursts > 0]
cols = ["n_bursts", "run_len_median", "run_len_max", "accel_z_at_burst", "accel_z_random", "ringdown_ratio"]
b[cols].describe(percentiles=[.5]).T[["count", "min", "50%", "max"]].round(2)

### Spot check of the threshold
A look at individual files, not an error-rate estimate. Train + val only: 15 randomly chosen files **flagged by the
threshold**, and every unflagged file whose peak robust z lies between 8 and the threshold (there are only 6, not 15).
For each, the largest microphone event ±2 ms, with the largest accelerometer robust z in the same window.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Picks the files, finds each one's largest microphone event, and measures it: how high it peaks (robust z), how many samples stay above half that peak, and how strongly the accelerometers move at the same moment. It plots the events in two grids, flagged and unflagged.</div>

In [ ]:
rng = np.random.default_rng(0)
flagged = trval[trval.peakz_mic > pre.mic.burst_z]
spot_flagged = flagged.iloc[np.sort(rng.choice(len(flagged), 15, replace=False))]
spot_unflagged = trval[(trval.peakz_mic >= 8) & (trval.peakz_mic <= pre.mic.burst_z)]


def event(rec_id: str) -> dict:
    x = load(rec_id)
    zm = robust_z(x[:, MIC])
    k = int(np.argmax(zm))
    sl = slice(max(k - half, 0), k + half)
    accel = max(robust_z(x[:, c])[sl].max() for c in ACCELS)
    return {"rec_id": rec_id, "peak_z": zm[k], "samples_above_half_peak": int((zm[sl] > zm[k] / 2).sum()),
            "accel_max_z_at_event": accel, "k": k}


def grid(files: pd.DataFrame, title: str, fname: str) -> pd.DataFrame:
    ev = pd.DataFrame([event(r) for r in files.rec_id]).merge(files[["rec_id", "class10", "severity"]])
    n = len(ev)
    fig, axes = plt.subplots(int(np.ceil(n / 5)), 5, figsize=(17, 2.6 * np.ceil(n / 5)), squeeze=False)
    for ax, r in zip(axes.flat, ev.itertuples()):
        m = load(r.rec_id)[:, MIC]
        sl = slice(max(r.k - half, 0), r.k + half)
        ax.plot((np.arange(m.size)[sl] - r.k) / FS * 1e3, m[sl], lw=0.7, color="#8172b2")
        ax.set_title(f"{r.rec_id}\nz {r.peak_z:.0f}, accel z {r.accel_max_z_at_event:.1f}", fontsize=7)
        ax.tick_params(labelsize=7)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(title)
    fig.supxlabel("time from event peak (ms)")
    fig.supylabel("microphone (V)")
    fig.tight_layout()
    fig.savefig(FIG / fname)
    return ev.drop(columns="k")


spot_f = grid(spot_flagged, "15 random files flagged by the threshold (train + val)", "02_mic_spot_flagged.png")
spot_u = grid(spot_unflagged, "Unflagged files with peak z 8–18.2 (train + val)", "02_mic_spot_unflagged.png")
display(spot_f.round(1))
spot_u.round(1)

**Spot check (visual, not an error-rate estimate).** 14 of the 15 flagged files show the same event: a single
sample down to −115…−205 V (robust z 600–1,300), symmetric ringing of about 0.3 ms around it, and little accelerometer
movement (largest robust z 1.9–6.7). The 15th, `overhang__cage_fault__35g__24.7808`, has the same shape at a much lower
level (−7 V, z 45). None of the 6 unflagged files looks like this: their largest events span several samples, reach at
most a few volts and do not ring. Two of them (`horizontal-misalignment__1.5mm__23.552`,
`underhang__outer_race__6g__24.1664`) coincide with large accelerometer movement (z 24–26), so they look like real
mechanical events that the microphone heard. On this sample, the threshold separates the two shapes.

### Residual ringing after the patch
Rule, fixed before looking: change `pad` only if the **cleaned** microphone, aligned on the flagged runs, stays more
than 1.2 × its random-position baseline beyond the patched samples and decays away from the run. Then `pad` becomes the
smallest value that brings it back within 1.2 ×. Profiles are the median |robust z| over every flagged run in train +
val, after the run (aligned on its last sample) and before it (aligned on its first sample, mirrored).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the burst-aligned profiles from the scan cache and plots, before and after each flagged run, how far the raw microphone and the cleaned microphone sit above their normal level. It also recomputes the smallest padding that clears the ringing, and checks it equals the value in the settings.</div>

In [ ]:
al = np.load(caches["micalign"])
lags, base = al["lags"], np.median(al["random"])
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), sharey=True)
needed = {}
for ax, (side, raw_k, clean_k) in zip(axes, [("after the run", "raw", "clean"), ("before the run", "raw_pre", "clean_pre")]):
    raw_p = np.median(al[raw_k], axis=0) / base
    clean_p = np.median(al[clean_k], axis=0) / base
    ax.semilogy(lags, raw_p, label="raw")
    ax.semilogy(lags, clean_p, label=f"cleaned (pad {pre.mic.burst_pad})")
    ax.axhline(1.2, color="red", ls="--", lw=0.8, label="1.2 × baseline")
    ax.axvline(pre.mic.burst_pad, color="k", lw=0.6)
    ax.set(xlabel="samples from the run edge", title=side, xlim=(-20, 60))
    above = lags[(lags > 0) & (raw_p > 1.2)]
    needed[side] = int(above.max()) if above.size else 0
axes[0].set_ylabel("median |robust z| / baseline")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "02_mic_residual_ringing.png")
print(f"{len(al['raw'])} flagged runs in {len(set(al['rec_id']))} files; median run length {np.median(al['run_len']):.0f} samples")
print(f"ringing above 1.2 × baseline for {needed} samples → pad {max(needed.values())}")
assert max(needed.values()) == pre.mic.burst_pad, "configs/base.yaml mic.burst_pad is out of date"

**Residual ringing → `pad` changed from 5 to 15.** With the original `pad = 5`, the cleaned microphone was still
7.8 × baseline one sample past the patch, 3.5 × at 10 samples and 2.3 × at 12. It decayed to within 1.2 × only after 15
samples, with the same profile before and after each run. That is visible, event-aligned residual ringing, and its
symmetry is what a linear-phase anti-alias filter produces when it rings on an impulse. With `pad = 15` the cleaned
profile stays within 1.15 × of baseline right next to the patch (plotted above), and the median post-burst RMS ratio
from the scan fell from 1.32 to 1.00.

Could the flagged files be compared with matched unflagged ones (same class, mass and similar speed)? Counts per
class and mass in train + val:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Counts, for each class and mass that has flagged files, how many train + val files are flagged and how many are not, to see whether there are enough matched controls for a fair comparison.</div>

In [ ]:
aff = trval[trval.class10.isin(flagged.class10.unique())]
pd.crosstab([aff.class10, aff.severity], aff.peakz_mic > pre.mic.burst_z).rename(
    columns={True: "flagged", False: "unflagged"}).query("flagged > 0")

**Matched controls are too few to conclude from a group comparison.** Where most files are flagged (imbalance 30 g
and 35 g, underhang cage 20 g and 35 g) there are only 7–11 unflagged train + val files per folder, before any matching
on speed. So no conclusion is drawn from comparing flagged and unflagged groups; the pad change rests only on the
aligned evidence above.

### Removing them
`ml.preprocess.remove_mic_bursts` flags samples above the threshold, widens each run by ±5 samples and interpolates
linearly across it. One of the bursty train files, before and after:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Runs the burst-removal function on one recording and plots the microphone before (grey) and after (purple): all 5 seconds on the left, a zoom on the largest burst on the right.</div>

In [ ]:
rid = burst_train[burst_train.class10 == "imbalance"].sort_values("peakz_mic").iloc[-10].rec_id
raw_mic = load(rid)[:, MIC]
clean_mic, n_b = remove_mic_bursts(raw_mic, pre.mic.burst_z, pre.mic.burst_pad)
k = int(np.argmax(np.abs(raw_mic - np.median(raw_mic))))
t = np.arange(raw_mic.size) / FS
fig, axes = plt.subplots(1, 2, figsize=(15, 3.4))
axes[0].plot(t, raw_mic, lw=0.3, color="#bbb", label="raw")
axes[0].plot(t, clean_mic, lw=0.3, color="#8172b2", label="cleaned")
axes[0].set(xlabel="time (s)", ylabel="microphone (V)", title=f"{rid}: {n_b} bursts removed")
axes[0].legend(loc="upper right")
sl = slice(k - 150, k + 150)
axes[1].plot(t[sl] * 1e3, raw_mic[sl], "o-", ms=2, lw=0.8, color="#bbb", label="raw")
axes[1].plot(t[sl] * 1e3, clean_mic[sl], lw=1.2, color="#8172b2", label="cleaned")
axes[1].set(xlabel="time (ms)", ylabel="microphone (V)", title="largest burst, zoomed", ylim=(-1.5, 1.5))
fig.tight_layout()
fig.savefig(FIG / "02_mic_before_after.png")
print(f"peak robust z: {robust_z(raw_mic).max():.0f} → {robust_z(clean_mic).max():.1f}")

### Shortcut test
Can a model name the class from microphone peak statistics alone (max |x|, peak robust z, crest factor, kurtosis per
recording)? RandomForest, 5-fold CV by `cv_fold` on train + val (folds are by recording), macro-F1 over 10 classes
(chance ≈ 0.10). Before vs after burst removal, and the burst flag on its own. Rule 10 bans these statistics as features in any case; this test only shows how much the bursts, and the cleaning, change what they reveal.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tests whether a model could guess the class from microphone spike statistics alone. It trains a random forest with 5-fold cross-validation, before and after burst removal, and reports the macro-F1 score (0.10 would be pure guessing). It also measures how reliably "this file has bursts" points to the two affected classes.</div>

In [ ]:
y = mic.class10.to_numpy()
feats = ["max_abs", "peakz", "crest", "kurtosis"]


def oof_predict(columns: list[str]) -> np.ndarray:
    X = mic[columns].to_numpy()
    pred = np.empty(len(mic), dtype=object)
    for f in range(5):
        tr, te = (mic.cv_fold != f).to_numpy(), (mic.cv_fold == f).to_numpy()
        pred[te] = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X[tr], y[tr]).predict(X[te])
    return pred


rows = {}
for name, cols in {"raw mic (4 stats)": [f"raw_{f}" for f in feats],
                   "cleaned mic (4 stats)": [f"clean_{f}" for f in feats],
                   "raw, shape only (peakz, crest, kurtosis)": [f"raw_{f}" for f in feats[1:]],
                   "cleaned, shape only": [f"clean_{f}" for f in feats[1:]]}.items():
    p = oof_predict(cols)
    per = f1_score(y, p, average=None, labels=CLASS10)
    rows[name] = {"macro-F1": f1_score(y, p, average="macro"),
                  "F1 imbalance": per[CLASS10.index("imbalance")],
                  "F1 underhang_cage": per[CLASS10.index("underhang_cage")]}
shortcut = pd.DataFrame(rows).T.round(3)
has = mic.n_bursts > 0
in_pair = mic.class10.isin(["imbalance", "underhang_cage"])
print(f"burst present → imbalance or underhang_cage: precision {precision_score(in_pair, has):.3f}, "
      f"recall {has[in_pair].mean():.3f} ({has.sum()} bursty of {in_pair.sum()} such recordings)")
shortcut

**Cleaning does not make the microphone statistics uninformative, and may leave its own trace.** After removal
(`pad = 15`), macro-F1 rises slightly, from 0.628 to 0.638, and underhang-cage F1 from 0.76 to 0.82. Replacing about
66 × 31 samples per flagged file with straight lines lowers exactly those files' crest factor and kurtosis, so the
cleaned statistics can still reveal that a file had bursts. This is one reason rule 10 bans microphone peak, crest and
kurtosis features outright, as a conservative policy. Whether the cleaned microphone representation used in Stage 2
(log-Mel) carries such a trace is **unresolved**, and should be tested there with the same shortcut test.

## (b) The one > 1 % speed disagreement
`overhang__cage_fault__6g__16.1792` (train): tacho edges, tacho FFT and the accelerometer comb disagree by 1.3 %.
The instantaneous speed from each tacho interval shows why.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Looks at the one recording whose three speed estimates disagree. It plots the speed revolution by revolution, the tacho spectrum, and the region around the 4th shaft harmonic, with each method's estimate drawn as a dashed line.</div>

In [ ]:
rid = "overhang__cage_fault__6g__16.1792"
row = meta.set_index("rec_id").loc[rid]
x = load(rid)
ti = tacho_intervals(x[:, 0], pre)
inst = FS / (ti.intervals / ti.revs)
t_mid = (ti.edges[:-1] + ti.intervals / 2) / FS
est = {"tacho_edges": row.fr_tacho_edges, "tacho_fft": row.fr_tacho_fft, "accel_comb": row.fr_accel_comb}
colors = {"tacho_edges": "green", "tacho_fft": "orange", "accel_comb": "purple"}

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].plot(t_mid, inst, "o-", ms=3, color="#333")
for k, v in est.items():
    axes[0].axhline(v, color=colors[k], ls="--", lw=1, label=f"{k} {v:.3f} Hz")
axes[0].set(xlabel="time (s)", ylabel="instantaneous shaft speed (Hz)", title="Speed per revolution")
axes[0].legend(fontsize=8)
f, a = amplitude_spectrum(x[:, 0], FS)
band = (f > 14.5) & (f < 17.5)
axes[1].plot(f[band], a[band], "o-", ms=3)
for k, v in est.items():
    axes[1].axvline(v, color=colors[k], ls="--", lw=1)
axes[1].set(xlabel="frequency (Hz)", ylabel="tacho amplitude (V)", title="Tacho spectrum (0.2 Hz bins)")
f, a = amplitude_spectrum(x[:, CH.index("over_radial")], FS)
band = (f > 3.8 * 15) & (f < 4.2 * 16.5)
axes[2].plot(f[band], a[band], lw=0.8)
for k, v in est.items():
    axes[2].axvline(4 * v, color=colors[k], ls="--", lw=1)
axes[2].set(xlabel="frequency (Hz)", ylabel="over_radial amplitude (V)", title="4th shaft harmonic region")
fig.tight_layout()
fig.savefig(FIG / "02_speed_disagreement.png")
print(f"instantaneous speed {inst.min():.2f}–{inst.max():.2f} Hz, mean {inst.mean():.3f}, std {inst.std():.3f} "
      f"({inst.std() / inst.mean():.1%}); clean intervals {ti.good.mean():.0%}")

**Verdict:** the speed wanders between about 15.5 and 16.4 Hz within the 5 s (±3 %), with every interval clean. The
tacho-edge estimate (15.754 Hz) is the true average over the recording, so the chosen speed is right. The FFT peak is
smeared over 15.4–16.2 Hz and the comb is pulled low by the smeared harmonics; neither is wrong, the recording is simply
non-stationary. Its order spectra will be blurred by about ±3 % per order. A per-recording speed-variability number
would find any others like it; that is a Stage 2 quality column, not done here.

## (c) Spectra

### Order spectra, one train recording per class
Frequency divided by measured shaft speed, so harmonics of the shaft sit on integers. One train recording per class,
nearest to 30 Hz, no quality flags. Imbalance should raise 1×; misalignment 2× (and axial energy); bearing faults barely
show here, because BPFO ≈ 3× and BPFI ≈ 5× coincide with shaft orders.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots vibration against shaft order (frequency divided by shaft speed) for one training recording per class and each of the six accelerometers. Red lines mark whole multiples of shaft speed, where imbalance and misalignment show up.</div>

In [ ]:
orders = np.arange(0, 10.5, 0.01)
fig, axes = plt.subplots(len(CLASS10), len(ACCELS), figsize=(18, 22), sharex=True)
for i, cls in enumerate(CLASS10):
    c = train[(train.class10 == cls) & train.quality_flags.isna()]
    r = c.loc[(c.fr_measured_hz - 30).abs().idxmin()]
    x = load(r.rec_id)
    for j, ch in enumerate(ACCELS):
        ax = axes[i, j]
        ax.semilogy(orders, order_spectrum(x[:, ch], FS, r.fr_measured_hz, orders), lw=0.6)
        for h in range(1, 11):
            ax.axvline(h, color="red", lw=0.4, alpha=0.4)
        if i == 0:
            ax.set_title(CH[ch])
        if j == 0:
            ax.set_ylabel(f"{cls}\n{r.fr_measured_hz:.1f} Hz\namp (V)", fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("order (× shaft speed)")
fig.tight_layout()
fig.savefig(FIG / "02_order_spectra.png")

### Mean amplitude spectrum per channel
Geometric mean of the Welch amplitude spectral density over a stratified train subset (15 recordings per class, bursts
removed from the microphone). This is where the ~23 kHz "resonance" of CLAUDE.md §3 should appear.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Averages the frequency content of each channel over 150 training recordings and plots it, with a zoom on 21.5–22.5 kHz. The table lists each channel's biggest peak above 5 kHz, how far it rises above the background and how narrow it is.</div>

In [ ]:
ms = np.load(caches["meanspec"], allow_pickle=True)
f, asd = ms["freqs"], ms["asd"]
gmean = np.exp(np.log(asd + 1e-12).mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [3, 1]})
for i, ch in enumerate(CH):
    axes[0].loglog(f[1:], gmean[i, 1:], lw=0.7, label=ch)
    axes[1].plot(f, gmean[i] / np.median(gmean[i][(f > 5000) & (f < 20000)]), lw=0.8, label=ch)
axes[0].set(xlabel="frequency (Hz)", ylabel="amplitude spectral density (V/√Hz)", title=f"Mean ASD, n={asd.shape[0]}")
axes[0].legend(fontsize=7, ncol=2)
axes[1].set(xlim=(21500, 22500), yscale="log", xlabel="frequency (Hz)", ylabel="relative to 5–20 kHz median",
            title="zoom 21.5–22.5 kHz")
fig.tight_layout()
fig.savefig(FIG / "02_mean_asd.png")

hf = f > 5000
peak_bins = []
for i, ch in enumerate(CH):
    j = int(np.argmax(gmean[i][hf]))
    base = np.median(gmean[i][(f > 5000) & (f < 20000)])
    width = int((gmean[i][hf] > gmean[i][hf][j] / 2).sum())
    peak_bins.append({"channel": ch, "peak above 5 kHz (Hz)": f[hf][j], "× 5–20 kHz median": gmean[i][hf][j] / base,
                      "bins above half max": width})
print(f"bin width {f[1]:.1f} Hz")
pd.DataFrame(peak_bins).round(1)

### Which accelerometer hears which bearing?
Before choosing a demodulation band per channel: high-pass each accelerometer above 1 kHz (bearing impacts live there,
shaft harmonics do not) and compare its RMS and kurtosis for train 0 g bearing faults against train normal. If the
labels of CLAUDE.md §3 are right, underhang faults should light up the `under_*` channels.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Compares high-frequency vibration (above 1 kHz) for each bearing fault against normal recordings, on every accelerometer. Numbers above 1 mean that sensor picks up the fault; darker cells mean a stronger response.</div>

In [ ]:
sensor = pd.read_parquet(caches["sensor"])
acc = [CH[c] for c in ACCELS]
rms = sensor.groupby("class10")[[f"hp_rms_{c}" for c in acc]].median()
rms.columns = acc
ratio = (rms / rms.loc["normal"]).round(1)
order_rows = ["normal", "underhang_ball", "underhang_cage", "underhang_outer", "overhang_ball", "overhang_cage",
              "overhang_outer"]
ratio = ratio.loc[order_rows]
print("median > 1 kHz RMS relative to train normal (n per class:",
      sensor.class10.value_counts().reindex(order_rows).to_dict(), ")")
ratio.style.background_gradient(cmap="Blues", axis=None, vmin=0, vmax=10).format("{:.1f}")

### Spectral kurtosis → demodulation band
Spectral kurtosis measures how impulsive each frequency band is (0 for Gaussian noise). Bearing impacts excite a band;
the envelope should be taken there. Fixed grid: widths 1, 2, 4 and 8 kHz, stepped by half a width, lower edge ≥ 1 kHz.
Computed on train 0 g outer-race recordings (both bearing positions) and train normal recordings.

Two things decide the band (physics and train data only):
- A band must be **at least 2 kHz wide**: the envelope has to carry up to 12 orders at 61 Hz (≥ 1.46 kHz).
- Normal machines are impulsive in some bands too, so the score is the **contrast**: median SK of outer-race files minus
  median SK of normal files. Each channel is scored against the bearing position it hears best (see the channel-group observation
  below), and the band with the largest contrast wins.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For each frequency band, measures how much more impulsive (spiky) outer-race recordings are than normal ones. It plots that contrast, then applies the project's band-selection rule (<code>ml.features.select_demod_band</code>) to the same training recordings and checks the result equals the band stored in the settings. It also checks that only <code>split_closed</code> train recordings were used.</div>

In [ ]:
sk = pd.read_parquet(caches["sk"])
sk["position"] = np.where(sk.class10 == "normal", "normal", sk.class10.str.split("_").str[0])
norm_sk = sk[sk.position == "normal"].groupby(["channel", "lo_hz", "hi_hz"]).sk.median()
contrast = (sk[sk.position != "normal"].groupby(["channel", "position", "lo_hz", "hi_hz"]).sk.median()
            - norm_sk).dropna().rename("contrast").reset_index()
contrast["width"] = contrast.hi_hz - contrast.lo_hz
contrast["centre"] = (contrast.hi_hz + contrast.lo_hz) / 2

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, ch in zip(axes, ["under_radial", "over_radial"]):
    for pos, ls in [("underhang", "-"), ("overhang", "--")]:
        for w in sorted(contrast.width.unique()):
            c = contrast[(contrast.channel == ch) & (contrast.position == pos) & (contrast.width == w)]
            ax.plot(c.centre / 1e3, c.contrast, ls, marker=".", lw=0.9, label=f"{pos} outer, {w / 1e3:g} kHz")
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, xlabel="band centre (kHz)")
axes[0].set_ylabel("median SK, outer race − normal")
axes[1].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG / "02_sk_bands.png")

assert set(sk.rec_id) <= set(train.rec_id), "band selection may only see split_closed train recordings"
chosen = pd.DataFrame(select_demod_band(train, sk, pre)).T
for ch, r in chosen.iterrows():
    assert tuple(pre.envelope.bands_hz[ch]) == (r.lo_hz, r.hi_hz), f"configs/base.yaml band for {ch} is out of date"
chosen[["fault_class", "lo_hz", "hi_hz", "contrast", "n_fault", "n_reference"]]

### Selected bands against the sensor ratings
The accelerometers are rated only up to a point. MAFAULDA lists both models but does not say which one is on which
bearing, so every band is checked against both.

| Model | ±3 dB range | Resonance | Source |
| --- | --- | --- | --- |
| IMI 604B31 (triaxial) | 0.5–5,000 Hz | 10 kHz | spec sheet Revision B, ECN 49214 |
| IMI 601A01 | 0.27–10,000 Hz | 16 kHz | spec sheet Revision G, ECN 14824 |

Above the ±3 dB range the sensitivity is not flat and, near resonance, strongly amplified. That does not make a band
useless (impacts excite resonances), but a feature built there depends on how each sensor is mounted.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Reads the band chosen for every evaluation track from <code>results/demod_bands.json</code> and, for each band, says whether it lies inside, partly inside or outside the rated range of each sensor model, and whether it covers that model's resonance.</div>

In [ ]:
bands = json.loads((SYSTEM_CODE_DIR / "results" / "demod_bands.json").read_text())["tracks"]


def rating(lo: float, hi: float, spec) -> str:
    r_lo, r_hi = spec.range_3db_hz
    where = "inside" if r_lo <= lo and hi <= r_hi else ("outside" if lo >= r_hi else "partly outside")
    return where + (" (covers resonance)" if lo <= spec.resonance_hz <= hi else "")


rows = []
for track, t in bands.items():
    for ch, b in t["channels"].items():
        rows.append({"track": track, "channel": ch, "band (kHz)": f"{b['lo_hz'] / 1e3:g}–{b['hi_hz'] / 1e3:g}",
                     **{f"vs {m}": rating(b["lo_hz"], b["hi_hz"], s) for m, s in pre.envelope.sensors.items()}})
ratings = pd.DataFrame(rows)
ratings.drop_duplicates(subset=["channel", "band (kHz)"]).merge(
    ratings.groupby(["channel", "band (kHz)"]).track.apply(lambda s: ", ".join(s)).rename("tracks"),
    on=["channel", "band (kHz)"]).drop(columns="track")

Does the out-of-range band actually carry more bearing evidence than a band the sensors are rated for? On
**train** recordings only (0 g bearing faults and normal), compare each fault's own comb (outer race → BPFO, ball →
2×BSF, cage → FTF) in the selected closed-track band against the a priori in-range band 1–5 kHz. Values are the median
comb prominence minus the normal median (dB), with a one-sided Mann–Whitney p-value against normal. This is a report;
the bands are not switched.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads envelope spectra computed in both bands for the same train recordings, scores each fault's own comb in each band, and shows the difference from normal recordings side by side, as a table and a bar chart.</div>

In [ ]:
bc = np.load(caches["bandcompare"], allow_pickle=True)
bo, bspec, bcls, bchan, bband = bc["orders"], bc["spec"], bc["class10"], list(bc["channels"]), list(bc["bands"])
lines = comb_lines(pre)
own = {"outer": "bpfo", "ball": "bsf", "cage": "ftf"}
rows = []
for i in range(len(bcls)):
    for j, ch in enumerate(bchan):
        for k, band in enumerate(bband):
            rows.append({"class10": bcls[i], "channel": ch, "band": band,
                         **{f: line_prominence_db(bo, bspec[i, j, k], L, pre.comb.tol_orders,
                                                  pre.comb.bg_halfwidth_orders).mean() for f, L in lines.items()}})
bs = pd.DataFrame(rows)
res = []
for (ch, band), g in bs.groupby(["channel", "band"]):
    normal = g[g.class10 == "normal"]
    for cls in sorted(set(g.class10) - {"normal"}):
        fam = own[cls.split("_")[1]]
        v = g.loc[g.class10 == cls, fam]
        res.append({"channel": ch, "class10": cls, "comb": fam, "band": band,
                    "excess dB": v.median() - normal[fam].median(),
                    "p": mannwhitneyu(v, normal[fam], alternative="greater").pvalue})
band_cmp = pd.DataFrame(res).pivot_table(index=["channel", "class10", "comb"], columns="band",
                                         values=["excess dB", "p"])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, ch in zip(axes, bchan):
    t = band_cmp.loc[ch, "excess dB"]
    t.plot.bar(ax=ax, color=["#bbb", "#4c72b0"], width=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, xlabel="", ylabel="own comb − normal median (dB)")
    ax.set_xticklabels([f"{c}\n({f})" for c, f in t.index], rotation=0, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "02_band_comparison.png")
band_cmp.round(3)

### Envelope spectra
Band-pass in the chosen band → Hilbert envelope → spectrum, in orders. Median over train + val 0 g recordings of each
class. Cursors: FTF (cage), 2×BSF (ball), BPFO (outer race), BPFI (inner race); faint grey lines are integer shaft
orders. Each channel is shown with the bearing faults of the position it hears best.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes the envelope spectrum (the rhythm of the impacts inside the chosen band) for normal, outer-race, ball and cage recordings, and plots the typical one per class. Coloured dashed lines mark where each type of bearing fault should produce a peak.</div>

In [ ]:
env = np.load(caches["envelope"], allow_pickle=True)
eo, espec, ecls, echan = env["orders"], env["spec"], env["class10"], list(env["channels"])
cursors = {"FTF": (pre.bearing.ftf, "#2ca02c"), "2×BSF": (2 * pre.bearing.bsf, "#9467bd"),
           "BPFO": (pre.bearing.bpfo, "#d62728"), "BPFI": (pre.bearing.bpfi, "#ff7f0e")}
hears = {"under_radial": "overhang", "over_radial": "underhang"}
kinds = [("normal", "normal"), ("outer", "outer"), ("ball", "ball"), ("cage", "cage")]

fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True)
for j, ch in enumerate(echan):
    for i, (label, kind) in enumerate(kinds):
        cls = "normal" if kind == "normal" else f"{hears[ch]}_{kind}"
        s = np.median(espec[ecls == cls, j], axis=0)
        ax = axes[i, j]
        ax.semilogy(eo, s, lw=0.6, color="#333")
        for h in range(1, 13):
            ax.axvline(h, color="#ccc", lw=0.5)
        for name, (o, col) in cursors.items():
            for h in (1, 2, 3):
                ax.axvline(h * o, color=col, lw=0.9, ls="--", alpha=0.8, label=name if (i, j, h) == (0, 0, 1) else None)
        ax.set(xlim=(0, 12), title=f"{ch} · {cls} (n={int((ecls == cls).sum())})")
        if j == 0:
            ax.set_ylabel("envelope amp (V)")
axes[0, 0].legend(fontsize=8, ncol=4)
for ax in axes[-1]:
    ax.set_xlabel("order (× shaft speed)")
fig.tight_layout()
fig.savefig(FIG / "02_envelope_spectra.png")

## (d) `cage_fault` audit
Is `cage_fault` a cage fault, or an inner-race flaw as CLAUDE.md §3 suspects? For every train + val 0 g recording, score
four combs in the envelope-order spectrum: how far each comb's lines stand above their local background (dB, mean over
lines), minus the median of normal recordings on the same channel.

| Comb | Lines (orders) |
| --- | --- |
| FTF (cage) | FTF × 1–5 |
| BSF (ball) | 2×BSF, 2×BSF ± FTF |
| BPFO (outer race) | BPFO × 1–3 |
| BPFI (inner race) | BPFI × 1–2, each ± 1× shaft sideband |

**Caveat:** BPFO (2.998) and BPFI (5.002) sit within 0.006 orders of integer shaft orders, finer than the spectrum
resolves, so the BPFO and BPFI combs also measure shaft-harmonic modulation. Only FTF and 2×BSF are free of integers.
Outer race and ball are the positive controls, normal the negative control. This is a finding only: no label or
parameter changes because of it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Scores every recording on four fault patterns (cage, ball, outer race, inner race): how far the expected peaks stand above their surroundings, minus the typical normal recording. The first table is the median score; the second is the share of recordings scoring higher than 95 % of normal ones.</div>

In [ ]:
lines = comb_lines(pre)
rows = []
for i in range(len(ecls)):
    if not (ecls[i] == "normal" or ecls[i].split("_")[0] in ("underhang", "overhang")):
        continue
    for j, ch in enumerate(echan):
        r = {"class10": ecls[i], "channel": ch}
        for fam, L in lines.items():
            r[fam] = line_prominence_db(eo, espec[i, j], L, pre.comb.tol_orders, pre.comb.bg_halfwidth_orders).mean()
        rows.append(r)
combs = pd.DataFrame(rows)
fams = list(lines)
ref = combs[combs.class10 == "normal"].groupby("channel")[fams]
excess = combs.copy()
excess[fams] = combs[fams] - ref.median().loc[combs.channel].to_numpy()
p95 = (combs[combs.class10 == "normal"].groupby("channel")[fams].quantile(0.95)
       - ref.median()).loc[combs.channel].to_numpy()
above = excess[fams].to_numpy() > p95

summary = excess.groupby(["channel", "class10"])[fams].median().round(2)
share = pd.DataFrame(above, columns=fams).assign(channel=excess.channel.values, class10=excess.class10.values)
share = share.groupby(["channel", "class10"])[fams].mean().round(2)
display(summary)
share

Median excess over normal (dB) per comb, on the channel that hears each bearing position, with Mann–Whitney p-values against normal:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws those scores as box plots for the channel that hears each bearing position, and tests whether each fault class scores higher than normal (small p-values mean it clearly does).</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
tests = []
for ax, ch in zip(axes, echan):
    classes = [f"{hears[ch]}_{k}" for k in ("outer", "ball", "cage")] + ["normal"]
    data = excess[excess.channel == ch]
    width = 0.2
    for k, fam in enumerate(fams):
        vals = [data.loc[data.class10 == c, fam] for c in classes]
        ax.boxplot(vals, positions=np.arange(len(classes)) + (k - 1.5) * width, widths=width * 0.9,
                   patch_artist=True, boxprops={"facecolor": f"C{k}", "alpha": 0.6}, showfliers=False,
                   medianprops={"color": "k"})
        for c, v in zip(classes[:-1], vals[:-1]):
            p = mannwhitneyu(v, data.loc[data.class10 == "normal", fam], alternative="greater").pvalue
            tests.append({"channel": ch, "class10": c, "comb": fam, "median excess dB": v.median(), "p (> normal)": p})
    ax.set_xticks(range(len(classes)), classes, rotation=15)
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, ylabel="comb prominence − normal median (dB)")
axes[1].legend([plt.Rectangle((0, 0), 1, 1, fc=f"C{k}", alpha=0.6) for k in range(4)], fams, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "02_cage_audit.png")
pd.DataFrame(tests).pivot_table(index=["channel", "class10"], columns="comb",
                                values=["median excess dB", "p (> normal)"]).round(3)

## (e) Other anomalies (exploratory; may stay unresolved)

### The 22 kHz tone, per recording
Amplitude of the line near 22,028 Hz and of its ±60 and ±120 Hz sidebands, per train + val recording and channel,
from one 5 s spectrum (0.2 Hz bins; the microphone with flagged events removed). If the tone's level followed class or
severity, a model could learn it as a shortcut.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tone amplitudes from the scan cache and plots, for four channels, the tone level of every recording grouped by class (log scale). The table gives the median level per class and severity on the underhang axial channel, the signal-to-floor ratio, and a Kruskal–Wallis test of whether the level differs between classes.</div>

In [ ]:
tone = pd.read_parquet(caches["tone"])
tone["snr_db"] = 20 * np.log10(tone["a+0"] / tone.floor)
show = ["under_axial", "under_radial", "over_axial", "mic"]
fig, axes = plt.subplots(1, len(show), figsize=(18, 4.2), sharey=True)
for ax, ch in zip(axes, show):
    d = tone[tone.channel == ch]
    ax.boxplot([d.loc[d.class10 == c, "a+0"] * 1e3 for c in CLASS10], showfliers=False)
    for k, c in enumerate(CLASS10):
        v = d.loc[d.class10 == c, "a+0"] * 1e3
        ax.scatter(np.full(len(v), k + 1) + np.random.default_rng(k).normal(0, 0.08, len(v)), v, s=3, alpha=0.4)
    p = kruskal(*[g["a+0"] for _, g in d.groupby("class10")]).pvalue
    ax.set(yscale="log", title=f"{ch} (Kruskal p = {p:.0e})")
    ax.set_xticks(range(1, len(CLASS10) + 1), CLASS10, rotation=60, fontsize=7)
axes[0].set_ylabel("22,028 Hz line amplitude (mV)")
fig.tight_layout()
fig.savefig(FIG / "02_tone_by_class.png")
print(f"line frequency: median {tone.f0_hz.median():.1f} Hz, IQR {tone.f0_hz.quantile(.25):.1f}–{tone.f0_hz.quantile(.75):.1f}")
print(f"±60 Hz sidebands / main line, median: {((tone['a-60'] + tone['a+60']) / 2 / tone['a+0']).median():.3f}")
ua = tone[tone.channel == "under_axial"]
ua.groupby(["class10", "severity"]).agg(n=("a+0", "size"), median_mV=("a+0", lambda s: s.median() * 1e3),
                                         median_snr_db=("snr_db", "median")).round(1)

**The tone is folder-linked, so it is a shortcut risk.** Whole folders are almost free of it (about 4–10 mV on
the underhang axial channel: every horizontal-misalignment severity, overhang ball 0 g, overhang outer 0 g, underhang
ball 35 g, underhang outer 6 g) while others carry hundreds of mV. That looks like acquisition sessions rather than
fault physics, and it is consistent with electrical interference. **Proposal, not applied:** before any representation
that reaches 22 kHz (spectrograms, log-Mel), remove the line and its sidebands, either with zero-phase IIR notches at
the recording's measured line frequency f0 and f0 ± 60, ± 120 Hz (Q ≈ f0 / 2 Hz, about 11,000, so each notch is ~2 Hz
wide) or by excluding 21.8–22.3 kHz from the representation. The envelope bands in use avoid it already
(the nearest, 23–25 kHz, starts above the +120 Hz sideband).

### Channel observation: a shared ~10 kHz peak?
The 604B31 is one triaxial housing with a 10 kHz mounted resonance; the three 601A01 units resonate at 16 kHz. If one
channel group shares a peak near 10 kHz in healthy recordings, that is consistent with that group being the 604B31. It
is supporting evidence at most: mounting and structure also make peaks.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Averages the frequency content of every accelerometer over all train normal recordings and lists, for each channel, the clearest peaks between 6 and 14 kHz with how far each rises above its surroundings.</div>

In [ ]:
tn_ids = train.loc[train.class10 == "normal", "rec_id"]
acc_names = [CH[c] for c in ACCELS]
asd_n = []
for rid in tn_ids:
    fw, p = welch(load(rid)[:, ACCELS].astype(np.float64), fs=FS, nperseg=4096, axis=0)
    asd_n.append(np.sqrt(p).T)
g_n = np.exp(np.log(np.array(asd_n) + 1e-15).mean(axis=0))
band = (fw >= 6000) & (fw <= 14000)
rows = []
for i, ch in enumerate(acc_names):
    pk, prop = find_peaks(np.log10(g_n[i][band]), prominence=0.15)
    for pr, f0 in sorted(zip(prop["prominences"], fw[band][pk]), reverse=True)[:3]:
        rows.append({"channel": ch, "peak (Hz)": int(f0), "prominence (× above surroundings)": round(10**pr, 2)})
print(f"{len(tn_ids)} train normal recordings")
pd.DataFrame(rows)

All three `under_*` channels have a peak at 9.2–10.0 kHz, weak but present (1.4–3 × above the surroundings); of
the `over_*` channels only radial does (9.4 kHz). That is consistent with the `under_*` group being the triaxial
604B31 with its 10 kHz resonance, but it does not prove it: the peaks are weak and structural resonances could produce
the same. **Unresolved observation.**

### Misalignment: which group responds?
MAFAULDA places the underhang bearing "between the rotor and the motor", so it is the one nearer the coupling where
misalignment acts. 1× and 2× shaft-order amplitude per accelerometer for train misalignment recordings, in dB relative
to the median of train normal recordings within ±3 Hz of the same speed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the 1× and 2× shaft-order amplitudes from the scan cache, compares each misalignment recording with normal recordings at a similar speed (within 3 Hz, at least 3 of them), and shows the median difference in dB per channel, plus the average of each channel group.</div>

In [ ]:
sh = pd.read_parquet(caches["shaft"])
nrm = sh[sh.class10 == "normal"]
rows = []
for r in sh[sh.class10 != "normal"].itertuples():
    ref = nrm[(nrm.fr_measured_hz - r.fr_measured_hz).abs() <= 3]
    if len(ref) < 3:
        continue
    row = {"class10": r.class10}
    for ch in acc_names:
        for h in ("1x", "2x"):
            row[f"{ch} {h}"] = 20 * np.log10(getattr(r, f"{ch}_{h}") / ref[f"{ch}_{h}"].median())
    rows.append(row)
mis = pd.DataFrame(rows)
print(f"{len(mis)} of {(sh.class10 != 'normal').sum()} misalignment recordings have ≥ 3 speed-matched normals")
out = {}
for h in ("1x", "2x"):
    t = mis.groupby("class10")[[f"{ch} {h}" for ch in acc_names]].median()
    t.columns = acc_names
    t["under_* mean"] = t[acc_names[:3]].mean(axis=1)
    t["over_* mean"] = t[acc_names[3:]].mean(axis=1)
    out[h] = t
pd.concat(out, names=["order"]).round(1)

Misalignment raises 1× and 2× mainly on the **overhang** channels (horizontal: over_radial 1× +9.7 dB; vertical:
over_tangential 2× +10.6 dB, over_radial 2× +5.4 dB), while the underhang channels stay within about ±2.6 dB. One overhang channel moves the other way: over_axial falls by 11.5 dB (1×) and 9.0 dB (2×) for vertical misalignment, which is not explained here. With the
underhang bearing nearer the coupling, that is consistent with the channel-group pattern the bearing
faults suggested, but vibration also travels through the rotor and frame, so it is not proof. **Unresolved
observation**; channel names are not changed.

## Findings
- **Microphone events flagged by the threshold are removed.** Threshold z = 18.2 (1.25 × the train-normal maximum of 14.57) sits in a clean gap (largest unflagged 14.57, smallest flagged 21.18) and flags 168 files (143 train+val), almost all imbalance 20–35 g and underhang cage 6–35 g. The flagged events look electrical (single samples to about −200 V, symmetric ~0.3 ms ringing, no accelerometer response); a spot check of 15 flagged and all 6 unflagged files with peak z 8–18.2 agrees. Event-aligned ringing lasted 15 samples either side, so `pad` went from 5 to 15. The flag is a precise but partial shortcut (99.3 % precision, 32 % recall), and cleaned mic peak statistics still give macro-F1 0.638 (0.628 raw), so cleaning may leave a trace; rule 10's feature ban stays as a conservative policy.
- **The ~23 kHz "resonance" is a narrowband line at 22,028 Hz with ±60/±120 Hz sidebands, consistent with electrical interference**, on every accelerometer and the microphone. Its level is folder-linked (whole folders near 4–10 mV, others 130–870 mV on the underhang axial channel), a shortcut risk; a notch is proposed, not applied.
- **Demodulation bands are chosen per evaluation track by `select_demod_band` on train data only** (`results/demod_bands.json`): under_radial 9–11 kHz and over_radial 13–17 kHz for the closed track and 7 LOCO folds; withholding an outer-race class moves one channel's band (1–3 kHz or 23–25 kHz, near-zero contrast). The healthy-only track uses 1–5 kHz, fixed a priori. The closed-track bands lie above the rated range of at least one sensor model and are **provisional**; on train data they beat the in-range 1–5 kHz band for underhang outer and overhang ball, but not for underhang ball.
- **Channel-group pattern: unresolved observation.** Bearing faults respond most on the opposite-named accelerometer group (underhang ball 11× on overhang axial and tangential, overhang ball 42× on underhang tangential), misalignment raises 1×/2× on the overhang channels (up to +10.6 dB), and all three `under_*` channels share a weak ~10 kHz peak like the 604B31's resonance. Channel names are unchanged.
- **`cage_fault`: no detectable inner-race signature under this analysis; the underhang-ball positive control failed, which limits sensitivity.** The BPFI comb is 1.3–2.4 dB below normal for both cage positions, while underhang outer lifts it by 10.2 dB; the cage (FTF) signature is weak and not significant. BPFO and BPFI sit on integer shaft orders, so these combs cannot separate outer from inner race. Labels are unchanged.
- **Speed**: the one > 1 % speed-disagreement file is non-stationary (15.47–16.35 Hz within 5 s); its chosen speed, the tacho-edge average 15.754 Hz, is right.

## Outputs written
- Figures: `results/figures/02_mic_peakz_hist.png`, `02_mic_burst_zoom.png`, `02_mic_before_after.png`,
  `02_speed_disagreement.png`, `02_order_spectra.png`, `02_mean_asd.png`, `02_sk_bands.png`,
  `02_envelope_spectra.png`, `02_cage_audit.png`
- `results/mic_burst_files.csv`: every file the burst threshold catches
- Parameters (in `configs/base.yaml` and `ml/config.py`, asserted above): `mic.burst_z`, `envelope.bands_hz`;
  comb definitions in `comb`